# Surface Code Threshold Estimation

This notebook estimates surface code error thresholds using PECOS native DEM generation and multiple decoder backends.

## Noise Models

We examine three progressively realistic noise models:

1. **Code Capacity**: Errors only on data qubits, perfect syndrome extraction
   - Expected threshold: ~11% for MWPM
   - Simplest model, upper bound on performance

2. **Phenomenological**: Data qubit errors + measurement errors
   - Expected threshold: ~3% for MWPM
   - Adds temporal dimension to matching

3. **Circuit-Level**: Full depolarizing noise on all operations
   - Expected threshold: ~0.5-1% for MWPM
   - Most realistic, includes error propagation through gates

## Threshold Estimation Method

The threshold is the physical error rate below which increasing code distance reduces logical error rate.

We estimate it by:
1. Running simulations at multiple physical error rates
2. Computing logical error rate for each (distance, error_rate) pair
3. Finding where curves for different distances cross

## Decoders

We compare multiple decoder backends via PECOS Rust bindings:
- **PyMatching**: Fast C++ MWPM decoder
- **FusionBlossom**: Pure Rust MWPM decoder  
- **BP+OSD**: Belief Propagation with Ordered Statistics Decoding
- **Tesseract**: A* search-based decoder

In [ ]:
import time
from collections.abc import Callable
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

# For Stim-based sampling (fast)
import stim
from pecos.qec.surface import (
    NoiseModel,
    SurfacePatch,
    generate_surface_code_dem,
)
from pecos.qec.surface.decode import generate_circuit_level_dem_from_builder
from pecos_rslib.decoders import PyMatchingDecoder, TesseractDecoder

## Configuration

In [ ]:
# Distances to test (odd only, starting at 9 to reduce finite-size effects)
DISTANCES = [9, 11, 13, 15]

# Number of shots per (distance, error_rate) pair
# More shots = better statistics but slower
NUM_SHOTS = 10000

# Number of syndrome rounds (typically d rounds for circuit-level)
def get_num_rounds(distance: int, noise_model: str) -> int:
    """Get number of syndrome rounds based on noise model."""
    if noise_model == "code_capacity":
        return 1  # Single round for code capacity
    return distance  # d rounds for phenomenological/circuit-level

# Decoders to compare
DECODERS = ["pymatching", "fusion_blossom"]  # Add more as needed

print(f"Distances: {DISTANCES}")
print(f"Shots per point: {NUM_SHOTS}")
print(f"Decoders: {DECODERS}")

## Helper Functions

In [ ]:
@dataclass
class ThresholdResult:
    """Results from threshold estimation."""
    noise_model: str
    decoder: str
    distances: list[int]
    error_rates: list[float]
    logical_error_rates: dict[int, list[float]]  # distance -> [ler per error_rate]
    num_shots: int
    elapsed_time: float


def sample_dem_with_stim(
    dem_string: str,
    num_shots: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Sample detection events and observable flips from a DEM using Stim.

    Args:
        dem_string: Detector Error Model in Stim format
        num_shots: Number of shots to sample

    Returns:
        (detection_events, observable_flips) arrays
    """
    dem = stim.DetectorErrorModel(dem_string)
    sampler = dem.compile_sampler()
    # Stim DEM sampler returns (detection_events, observable_flips, errors)
    detection_events, observable_flips, _ = sampler.sample(num_shots)
    return detection_events, observable_flips


def decode_samples(
    detection_events: np.ndarray,
    observable_flips: np.ndarray,
    decoder,
    decoder_type: str,
) -> int:
    """Decode sampled detection events and count logical errors.

    Args:
        detection_events: Array of detection events (num_shots, num_detectors)
        observable_flips: Array of true observable flips (num_shots, num_observables)
        decoder: Decoder instance
        decoder_type: Type of decoder for dispatch

    Returns:
        Number of logical errors
    """
    num_errors = 0
    num_shots = detection_events.shape[0]

    for i in range(num_shots):
        events = detection_events[i]
        true_flip = observable_flips[i, 0] if observable_flips.shape[1] > 0 else 0

        if decoder_type == "pymatching":
            result = decoder.decode(events.astype(np.uint8).tolist())
            predicted_flip = result.correction[0] if len(result.correction) > 0 else 0
        elif decoder_type == "fusion_blossom":
            result = decoder.decode(events.astype(np.uint8).tolist())
            predicted_flip = result.correction[0] if len(result.correction) > 0 else 0
            decoder.clear()
        elif decoder_type == "tesseract":
            detection_indices = [j for j, v in enumerate(events) if v]
            result = decoder.decode(detection_indices)
            predicted_flip = result.observables_mask & 1
        elif decoder_type == "bp_osd":
            result = decoder.decode(events.astype(np.uint8).tolist())
            predicted_flip = result.decoding[0] if len(result.decoding) > 0 else 0
        else:
            msg = f"Unknown decoder type: {decoder_type}"
            raise ValueError(msg)

        # Logical error if prediction doesn't match true flip
        if predicted_flip != true_flip:
            num_errors += 1

    return num_errors


def create_decoder_from_dem(dem_string: str, decoder_type: str):
    """Create a decoder from a DEM string."""
    # Filter logical_observable for Tesseract
    if decoder_type == "tesseract":
        dem_filtered = "\n".join(
            line for line in dem_string.split("\n")
            if not line.startswith("logical_observable")
        )
        return TesseractDecoder.from_dem(dem_filtered, preset="fast")

    if decoder_type == "pymatching":
        return PyMatchingDecoder.from_dem(dem_string)

    if decoder_type == "fusion_blossom":
        # FusionBlossom doesn't have from_dem, use PyMatching as fallback
        return PyMatchingDecoder.from_dem(dem_string)

    if decoder_type == "bp_osd":
        # BP+OSD doesn't directly support DEM format
        msg = "BP+OSD from DEM not yet implemented"
        raise NotImplementedError(msg)

    msg = f"Unknown decoder type: {decoder_type}"
    raise ValueError(msg)

In [ ]:
def run_threshold_sweep(
    noise_model: str,
    distances: list[int],
    error_rates: list[float],
    decoder_type: str,
    num_shots: int,
    dem_generator: Callable,
) -> ThresholdResult:
    """Run threshold estimation sweep.

    Args:
        noise_model: Name of noise model ('code_capacity', 'phenomenological', 'circuit_level')
        distances: List of code distances to test
        error_rates: List of physical error rates to sweep
        decoder_type: Decoder backend to use
        num_shots: Shots per (distance, error_rate) pair
        dem_generator: Function(patch, num_rounds, p) -> dem_string

    Returns:
        ThresholdResult with logical error rates
    """
    start_time = time.time()
    logical_error_rates = {d: [] for d in distances}

    total_points = len(distances) * len(error_rates)
    point = 0

    for d in distances:
        patch = SurfacePatch.create(distance=d)
        num_rounds = get_num_rounds(d, noise_model)

        for p in error_rates:
            point += 1

            # Generate DEM
            dem_string = dem_generator(patch, num_rounds, p)

            # Create decoder
            try:
                decoder = create_decoder_from_dem(dem_string, decoder_type)
            except (ValueError, NotImplementedError, RuntimeError) as e:
                print(f"  Decoder creation failed for d={d}, p={p}: {e}")
                logical_error_rates[d].append(float("nan"))
                continue

            # Sample and decode
            try:
                detection_events, observable_flips = sample_dem_with_stim(dem_string, num_shots)
                num_errors = decode_samples(detection_events, observable_flips, decoder, decoder_type)
                ler = num_errors / num_shots
            except (ValueError, RuntimeError, IndexError) as e:
                print(f"  Sampling/decoding failed for d={d}, p={p}: {e}")
                ler = float("nan")

            logical_error_rates[d].append(ler)

            print(f"  [{point}/{total_points}] d={d}, p={p:.4f}: LER={ler:.4f} ({num_errors}/{num_shots})")

    elapsed = time.time() - start_time

    return ThresholdResult(
        noise_model=noise_model,
        decoder=decoder_type,
        distances=distances,
        error_rates=error_rates,
        logical_error_rates=logical_error_rates,
        num_shots=num_shots,
        elapsed_time=elapsed,
    )

In [ ]:
def plot_threshold_result(result: ThresholdResult, ax=None, title=None):
    """Plot threshold estimation results."""
    if ax is None:
        _fig, ax = plt.subplots(figsize=(10, 7))

    colors = plt.cm.viridis(np.linspace(0, 0.8, len(result.distances)))
    markers = ["o", "s", "^", "D", "v", "<", ">", "p"]

    for i, d in enumerate(result.distances):
        lers = result.logical_error_rates[d]
        # Filter out NaN values
        valid_mask = ~np.isnan(lers)
        valid_ps = np.array(result.error_rates)[valid_mask]
        valid_lers = np.array(lers)[valid_mask]

        if len(valid_lers) > 0:
            ax.plot(valid_ps, valid_lers,
                   f"{markers[i % len(markers)]}-",
                   color=colors[i],
                   label=f"d={d}",
                   markersize=8,
                   linewidth=2)

    ax.set_xlabel("Physical Error Rate", fontsize=14)
    ax.set_ylabel("Logical Error Rate", fontsize=14)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, which="both")
    ax.set_ylim(1e-4, 1.0)

    if title:
        ax.set_title(title, fontsize=14)
    else:
        ax.set_title(f"{result.noise_model} - {result.decoder}\n({result.num_shots} shots, {result.elapsed_time:.1f}s)", fontsize=14)

    return ax


def estimate_threshold_crossing(result: ThresholdResult) -> float | None:
    """Estimate threshold from curve crossings.

    Finds approximate crossing point of adjacent distance curves.
    Returns average of crossing points.
    """
    crossings = []

    for i in range(len(result.distances) - 1):
        d1, d2 = result.distances[i], result.distances[i + 1]
        lers1 = np.array(result.logical_error_rates[d1])
        lers2 = np.array(result.logical_error_rates[d2])
        ps = np.array(result.error_rates)

        # Find where curves cross (sign change in difference)
        diff = lers1 - lers2
        valid_mask = ~(np.isnan(lers1) | np.isnan(lers2))
        diff = diff[valid_mask]
        ps_valid = ps[valid_mask]

        for j in range(len(diff) - 1):
            if diff[j] * diff[j + 1] < 0:  # Sign change
                # Linear interpolation
                p_cross = ps_valid[j] + (ps_valid[j + 1] - ps_valid[j]) * abs(diff[j]) / (abs(diff[j]) + abs(diff[j + 1]))
                crossings.append(p_cross)

    if crossings:
        return np.mean(crossings)
    return None

## Part 1: Code Capacity Threshold

In the code capacity model:
- Errors occur only on data qubits (bit-flip or phase-flip)
- Syndrome measurements are perfect
- Single syndrome round

This is equivalent to a phenomenological model with `p_meas = 0`.

Expected threshold: ~10.9% for MWPM on rotated surface code

In [ ]:
def generate_code_capacity_dem(patch: SurfacePatch, num_rounds: int, p: float) -> str:
    """Generate a code-capacity DEM (data errors only, perfect measurements)."""
    # Use phenomenological DEM with p_meas=0
    noise = NoiseModel(p1=0, p2=p, p_meas=0, p_init=0)
    return generate_surface_code_dem(patch, num_rounds=1, noise=noise, stab_type="Z")

# Test DEM generation
patch_test = SurfacePatch.create(distance=5)
dem_test = generate_code_capacity_dem(patch_test, 1, 0.05)
print("Code capacity DEM preview:")
for line in dem_test.split("\n")[:10]:
    print(f"  {line}")

In [ ]:
# Error rates to sweep for code capacity (around expected threshold ~11%)
# Note: Below threshold, larger distance = LOWER error rate (good)
#       Above threshold, larger distance = HIGHER error rate (bad)
CODE_CAPACITY_ERROR_RATES = [0.08, 0.09, 0.10, 0.105, 0.11, 0.115, 0.12, 0.13, 0.14]

print("=" * 60)
print("CODE CAPACITY THRESHOLD ESTIMATION")
print("=" * 60)

code_capacity_result = run_threshold_sweep(
    noise_model="code_capacity",
    distances=DISTANCES,
    error_rates=CODE_CAPACITY_ERROR_RATES,
    decoder_type="pymatching",
    num_shots=NUM_SHOTS,
    dem_generator=generate_code_capacity_dem,
)

threshold = estimate_threshold_crossing(code_capacity_result)
print(f"\nEstimated code capacity threshold: {threshold:.4f}" if threshold else "\nCould not estimate threshold")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_threshold_result(code_capacity_result, ax=ax)

threshold = estimate_threshold_crossing(code_capacity_result)
if threshold:
    ax.axvline(x=threshold, color="red", linestyle="--", alpha=0.7, label=f"Threshold ~ {threshold:.3f}")
    ax.legend()

ax.set_title(f"Code Capacity Threshold (PyMatching)\nEstimated: {threshold:.3f}" if threshold else "Code Capacity Threshold (PyMatching)", fontsize=14)
plt.tight_layout()
plt.show()

## Part 2: Phenomenological Threshold

In the phenomenological model:
- Errors occur on data qubits with probability `p`
- Measurement errors occur with probability `p`
- Multiple syndrome rounds (typically `d` rounds)
- No error propagation through gates

Expected threshold: ~2.9% for MWPM

In [ ]:
def generate_phenomenological_dem(patch: SurfacePatch, num_rounds: int, p: float) -> str:
    """Generate a phenomenological DEM (data + measurement errors)."""
    noise = NoiseModel(p1=0, p2=p, p_meas=p, p_init=0)
    return generate_surface_code_dem(patch, num_rounds=num_rounds, noise=noise, stab_type="Z")

# Test DEM generation
dem_test = generate_phenomenological_dem(patch_test, num_rounds=5, p=0.01)
print(f"Phenomenological DEM: {len(dem_test.split(chr(10)))} lines")

In [ ]:
# Error rates to sweep for phenomenological (around expected threshold ~3%)
PHENOM_ERROR_RATES = [0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045, 0.05]

print("=" * 60)
print("PHENOMENOLOGICAL THRESHOLD ESTIMATION")
print("=" * 60)

phenomenological_result = run_threshold_sweep(
    noise_model="phenomenological",
    distances=DISTANCES,
    error_rates=PHENOM_ERROR_RATES,
    decoder_type="pymatching",
    num_shots=NUM_SHOTS,
    dem_generator=generate_phenomenological_dem,
)

threshold = estimate_threshold_crossing(phenomenological_result)
print(f"\nEstimated phenomenological threshold: {threshold:.4f}" if threshold else "\nCould not estimate threshold")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_threshold_result(phenomenological_result, ax=ax)

threshold = estimate_threshold_crossing(phenomenological_result)
if threshold:
    ax.axvline(x=threshold, color="red", linestyle="--", alpha=0.7, label=f"Threshold ~ {threshold:.3f}")
    ax.legend()

ax.set_title(f"Phenomenological Threshold (PyMatching)\nEstimated: {threshold:.3f}" if threshold else "Phenomenological Threshold (PyMatching)", fontsize=14)
plt.tight_layout()
plt.show()

## Part 3: Circuit-Level Threshold

In the circuit-level model:
- Depolarizing errors after each gate
- Errors propagate through CNOT gates
- Measurement and initialization errors
- Hook errors from syndrome extraction circuit

This uses PECOS native DEM generation via:
```
TickCircuit -> DagCircuit -> DagFaultAnalyzer -> DemBuilder -> DEM
```

Expected threshold: ~0.5-1% for MWPM

In [ ]:
def generate_circuit_level_dem(patch: SurfacePatch, num_rounds: int, p: float) -> str:
    """Generate a circuit-level DEM using PECOS native fault propagation."""
    # Use same error rate for all noise sources (standard depolarizing)
    noise = NoiseModel(p1=p, p2=p, p_meas=p, p_init=p)
    return generate_circuit_level_dem_from_builder(patch, num_rounds=num_rounds, noise=noise, basis="Z")

# Test DEM generation
dem_test = generate_circuit_level_dem(patch_test, num_rounds=5, p=0.001)
print(f"Circuit-level DEM: {len(dem_test.split(chr(10)))} lines")
print("\nPreview:")
for line in dem_test.split("\n")[:10]:
    print(f"  {line}")

In [ ]:
# Error rates to sweep for circuit-level (around expected threshold ~0.5-1%)
CIRCUIT_ERROR_RATES = [0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.012]

print("=" * 60)
print("CIRCUIT-LEVEL THRESHOLD ESTIMATION (PECOS Native DEM)")
print("=" * 60)

circuit_level_result = run_threshold_sweep(
    noise_model="circuit_level",
    distances=DISTANCES,
    error_rates=CIRCUIT_ERROR_RATES,
    decoder_type="pymatching",
    num_shots=NUM_SHOTS,
    dem_generator=generate_circuit_level_dem,
)

threshold = estimate_threshold_crossing(circuit_level_result)
print(f"\nEstimated circuit-level threshold: {threshold:.4f}" if threshold else "\nCould not estimate threshold")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_threshold_result(circuit_level_result, ax=ax)

threshold = estimate_threshold_crossing(circuit_level_result)
if threshold:
    ax.axvline(x=threshold, color="red", linestyle="--", alpha=0.7, label=f"Threshold ~ {threshold:.4f}")
    ax.legend()

ax.set_title(f"Circuit-Level Threshold (PECOS Native DEM + PyMatching)\nEstimated: {threshold:.4f}" if threshold else "Circuit-Level Threshold (PECOS Native DEM)", fontsize=14)
plt.tight_layout()
plt.show()

## Part 4: All Thresholds Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

results = [
    (code_capacity_result, "Code Capacity"),
    (phenomenological_result, "Phenomenological"),
    (circuit_level_result, "Circuit-Level"),
]

for ax, (result, name) in zip(axes, results, strict=False):
    plot_threshold_result(result, ax=ax)
    threshold = estimate_threshold_crossing(result)
    if threshold:
        ax.axvline(x=threshold, color="red", linestyle="--", alpha=0.7)
        ax.set_title(f"{name}\nThreshold ~ {threshold:.4f}", fontsize=14)
    else:
        ax.set_title(name, fontsize=14)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 60)
print("THRESHOLD SUMMARY")
print("=" * 60)
print(f"{'Noise Model':<20} {'Threshold':<15} {'Expected':<15}")
print("-" * 50)
for result, name in results:
    threshold = estimate_threshold_crossing(result)
    expected = {"Code Capacity": "~10.9%", "Phenomenological": "~2.9%", "Circuit-Level": "~0.5-1%"}[name]
    thresh_str = f"{threshold:.4f}" if threshold else "N/A"
    print(f"{name:<20} {thresh_str:<15} {expected:<15}")

## Part 5: Decoder Comparison (Circuit-Level)

Compare different decoders on circuit-level noise model.

In [ ]:
# Use fewer distances for decoder comparison (faster)
DECODER_COMPARISON_DISTANCES = [9, 11, 13]
DECODER_COMPARISON_SHOTS = 5000

decoder_results = {}

for decoder_type in ["pymatching", "tesseract"]:
    print(f"\n{'=' * 60}")
    print(f"DECODER: {decoder_type.upper()}")
    print("=" * 60)

    try:
        result = run_threshold_sweep(
            noise_model="circuit_level",
            distances=DECODER_COMPARISON_DISTANCES,
            error_rates=CIRCUIT_ERROR_RATES,
            decoder_type=decoder_type,
            num_shots=DECODER_COMPARISON_SHOTS,
            dem_generator=generate_circuit_level_dem,
        )
        decoder_results[decoder_type] = result

        threshold = estimate_threshold_crossing(result)
        if threshold:
            print(f"\nEstimated threshold ({decoder_type}): {threshold:.4f}")
        else:
            print(f"\nCould not estimate threshold ({decoder_type})")
    except (ValueError, NotImplementedError, RuntimeError) as e:
        print(f"Error with {decoder_type}: {e}")

In [ ]:
if decoder_results:
    fig, axes = plt.subplots(1, len(decoder_results), figsize=(7 * len(decoder_results), 6))
    if len(decoder_results) == 1:
        axes = [axes]

    for ax, (decoder_type, result) in zip(axes, decoder_results.items(), strict=False):
        plot_threshold_result(result, ax=ax)
        threshold = estimate_threshold_crossing(result)
        if threshold:
            ax.axvline(x=threshold, color="red", linestyle="--", alpha=0.7)
            ax.set_title(f"{decoder_type}\nThreshold ~ {threshold:.4f}", fontsize=14)
        else:
            ax.set_title(decoder_type, fontsize=14)

    plt.tight_layout()
    plt.show()

    # Summary
    print("\n" + "=" * 60)
    print("DECODER COMPARISON SUMMARY (Circuit-Level)")
    print("=" * 60)
    print(f"{'Decoder':<20} {'Threshold':<15} {'Time (s)':<15}")
    print("-" * 50)
    for decoder_type, result in decoder_results.items():
        threshold = estimate_threshold_crossing(result)
        thresh_str = f"{threshold:.4f}" if threshold else "N/A"
        print(f"{decoder_type:<20} {thresh_str:<15} {result.elapsed_time:<15.1f}")

## Summary

This notebook estimated surface code thresholds for three noise models:

| Noise Model | Description | Expected Threshold |
|-------------|-------------|--------------------|
| Code Capacity | Data errors only, perfect measurements | ~10.9% |
| Phenomenological | Data + measurement errors | ~2.9% |
| Circuit-Level | Full depolarizing noise | ~0.5-1% |

**Key observations:**

1. **DEM Generation**: Circuit-level DEMs use PECOS native fault propagation:
   ```
   TickCircuit -> DagCircuit -> DagFaultAnalyzer -> DemBuilder -> DEM
   ```

2. **Sampling**: DEMs are sampled using Stim's efficient sampler

3. **Decoding**: Multiple decoder backends available via PECOS Rust bindings

4. **Threshold estimation**: Found by identifying crossing points of logical error rate curves

**For more accurate threshold estimation:**
- Use more shots (100k+)
- Use larger distances (up to 21+)
- Use finer error rate grid near threshold
- Apply proper statistical analysis (e.g., fitting to threshold model)